<a href="https://colab.research.google.com/github/muslimuddin2002/my-cse-projects/blob/main/python_Student_Management_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Student Management System
import json
import os
import re
from datetime import datetime

# ================= STUDENT CLASS =================
class Student:

    def __init__(
        self,
        student_id,
        name,
        father_name,
        mother_name,
        father_profession,
        mother_profession,
        blood_group,
        hobby,
        nid_birth,
        nid_type,  # NEW: "NID" or "Birth Certificate"
        date_of_birth,
        department,
        address,
        student_phone,
        parent_phone,
        email,
        cgpa
    ):

        self.student_id = student_id
        self.name = name
        self.father_name = father_name
        self.mother_name = mother_name
        self.father_profession = father_profession
        self.mother_profession = mother_profession
        self.blood_group = blood_group
        self.hobby = hobby
        self.nid_birth = nid_birth
        self.nid_type = nid_type  # Store which type
        self.date_of_birth = date_of_birth
        self.department = department
        self.address = address
        self.student_phone = student_phone
        self.parent_phone = parent_phone
        self.email = email
        self.cgpa = cgpa

    # Convert object to dictionary
    def to_dict(self):

        return {
            "student_id": self.student_id,
            "name": self.name,
            "father_name": self.father_name,
            "mother_name": self.mother_name,
            "father_profession": self.father_profession,
            "mother_profession": self.mother_profession,
            "blood_group": self.blood_group,
            "hobby": self.hobby,
            "nid_birth": self.nid_birth,
            "nid_type": self.nid_type,
            "date_of_birth": self.date_of_birth,
            "department": self.department,
            "address": self.address,
            "student_phone": self.student_phone,
            "parent_phone": self.parent_phone,
            "email": self.email,
            "cgpa": self.cgpa
        }


# ================= MAIN SYSTEM =================
class StudentManagementSystem:

    def __init__(self):

        self.students = []
        self.filename = "students.json"

        self.load_data()

    # ---------------- VALIDATION FUNCTIONS ----------------

    def validate_email(self, email):
        """Check if email is valid and unique"""
        # Check email format
        if not re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', email):
            return False, "Invalid email format! Must contain @ and domain (e.g., student@example.com)"

        # Check if email already exists
        for student in self.students:
            if student.email.lower() == email.lower():
                return False, "This email is already registered! Use a different email."

        return True, "Valid"

    def validate_phone(self, phone):
        """Check if phone contains only numbers"""
        if not phone.isdigit():
            return False, "Phone number must contain only digits (0-9)"
        if len(phone) < 10 or len(phone) > 15:
            return False, "Phone number must be 10-15 digits long"
        return True, "Valid"

    def validate_date_of_birth(self, date_str):
        """Validate date format: DD/MM/YYYY or DD-MM-YYYY or YYYY-MM-DD"""
        # Try different formats
        formats = ["%d/%m/%Y", "%d-%m-%Y", "%Y-%m-%d", "%d.%m.%Y"]

        for fmt in formats:
            try:
                date_obj = datetime.strptime(date_str, fmt)
                # Check if date is not in future
                if date_obj > datetime.now():
                    return False, "Date of birth cannot be in the future!"
                # Check if age is reasonable (5 to 100 years)
                age = datetime.now().year - date_obj.year
                if age < 5 or age > 100:
                    return False, f"Age ({age} years) seems invalid! Please check the date."
                return True, date_obj.strftime("%d/%m/%Y")  # Standard format
            except ValueError:
                continue

        return False, "Invalid date format! Use DD/MM/YYYY, DD-MM-YYYY, or YYYY-MM-DD"

    def validate_cgpa(self, cgpa):
        """Validate CGPA range"""
        if cgpa < 0 or cgpa > 4.00:
            return False, "CGPA must be between 0.00 and 4.00"
        return True, "Valid"

    def validate_student_id(self, student_id):
        """Check if student ID is unique"""
        for student in self.students:
            if student.student_id == student_id:
                return False, "Student ID already exists!"
        return True, "Valid"

    def validate_nid_birth(self, nid_number, nid_type):
        """Validate NID or Birth Certificate number"""
        if not nid_number.isdigit():
            return False, f"{nid_type} number must contain only digits!"

        if nid_type == "NID":
            if len(nid_number) < 10 or len(nid_number) > 17:
                return False, "NID number must be 10-17 digits long!"
        else:  # Birth Certificate
            if len(nid_number) < 10 or len(nid_number) > 20:
                return False, "Birth Certificate number must be 10-20 digits long!"

        return True, "Valid"

    # ---------------- ADD STUDENT ----------------
    def add_student(self):

        print("\n========== ADD NEW STUDENT ==========\n")

        try:
            # Student ID (Unique)
            while True:
                student_id = input("1. Enter Student ID: ")
                is_valid, message = self.validate_student_id(student_id)
                if is_valid:
                    break
                print(f"❌ {message}")

            name = input("2. Enter Student Name: ").strip()
            father_name = input("3. Enter Father's Name: ").strip()
            mother_name = input("4. Enter Mother's Name: ").strip()

            father_profession = input("5. Enter Father's Profession: ").strip()
            mother_profession = input("6. Enter Mother's Profession: ").strip()

            blood_group = input("7. Enter Blood Group (A+, B+, O+, AB+ etc.): ").strip()
            hobby = input("8. Enter Favourite Hobby: ").strip()

            # NID or Birth Certificate Selection
            print("\n9. Select Document Type:")
            print("   1. NID Card")
            print("   2. Birth Certificate")
            while True:
                doc_choice = input("   Choose (1 or 2): ").strip()
                if doc_choice == "1":
                    nid_type = "NID"
                    doc_name = "NID Number"
                    break
                elif doc_choice == "2":
                    nid_type = "Birth Certificate"
                    doc_name = "Birth Certificate Number"
                    break
                else:
                    print("❌ Invalid choice! Please enter 1 or 2.")

            # Validate NID/Birth number
            while True:
                nid_birth = input(f"9. Enter {doc_name}: ").strip()
                is_valid, message = self.validate_nid_birth(nid_birth, nid_type)
                if is_valid:
                    break
                print(f"❌ {message}")

            # Date of Birth with format validation
            print("\n10. Enter Date of Birth (Format: DD/MM/YYYY or DD-MM-YYYY):")
            print("    Example: 15/03/2000 or 15-03-2000")
            while True:
                date_of_birth = input("    Enter Date: ").strip()
                is_valid, result = self.validate_date_of_birth(date_of_birth)
                if is_valid:
                    date_of_birth = result  # Standardized format
                    break
                print(f"❌ {result}")

            department = input("11. Enter Department: ").strip()
            address = input("12. Enter Address: ").strip()

            # Student Phone validation
            print("\n13. Enter Student Phone Number (only digits):")
            while True:
                student_phone = input("    Phone: ").strip()
                is_valid, message = self.validate_phone(student_phone)
                if is_valid:
                    break
                print(f"❌ {message}")

            # Parent Phone validation
            print("\n14. Enter Parent Phone Number (only digits):")
            while True:
                parent_phone = input("    Phone: ").strip()
                is_valid, message = self.validate_phone(parent_phone)
                if is_valid:
                    break
                print(f"❌ {message}")

            # Email validation (format + unique)
            print("\n15. Enter Email Address:")
            while True:
                email = input("    Email: ").strip()
                is_valid, message = self.validate_email(email)
                if is_valid:
                    break
                print(f"❌ {message}")

            # CGPA validation
            while True:
                try:
                    cgpa = float(input("16. Enter CGPA (0.00 - 4.00): "))
                    is_valid, message = self.validate_cgpa(cgpa)
                    if is_valid:
                        break
                    print(f"❌ {message}")
                except ValueError:
                    print("❌ Invalid input! Please enter a number (e.g., 3.50)")

            # Create object with new parameter
            student = Student(
                student_id,
                name,
                father_name,
                mother_name,
                father_profession,
                mother_profession,
                blood_group,
                hobby,
                nid_birth,
                nid_type,  # New parameter
                date_of_birth,
                department,
                address,
                student_phone,
                parent_phone,
                email,
                cgpa
            )

            self.students.append(student)

            print("\n✅ Student Added Successfully!")

        except Exception as e:
            print(f"❌ Error adding student: {e}")

    # ---------------- VIEW STUDENTS ----------------
    def view_students(self):

        if len(self.students) == 0:

            print("\n❌ No student data found.")
            return

        print("\n========== ALL STUDENTS ==========\n")

        serial = 1

        for student in self.students:

            print(f"""
================ STUDENT {serial} ================

Student ID          : {student.student_id}
Student Name        : {student.name}

Father's Name       : {student.father_name}
Mother's Name       : {student.mother_name}

Father Profession   : {student.father_profession}
Mother Profession   : {student.mother_profession}

Blood Group         : {student.blood_group}

Favourite Hobby     : {student.hobby}

{student.nid_type}   : {student.nid_birth}

Date of Birth       : {student.date_of_birth}

Department          : {student.department}

Address             : {student.address}

Student Phone       : {student.student_phone}

Parent Phone        : {student.parent_phone}

Email               : {student.email}

CGPA                : {student.cgpa}

==================================================
""")

            serial += 1

    # ---------------- SEARCH STUDENT ----------------
    def search_student(self):

        search_id = input("Enter Student ID: ")

        for student in self.students:

            if student.student_id == search_id:

                print(f"""
========== STUDENT FOUND ==========

Student ID          : {student.student_id}
Student Name        : {student.name}

Father's Name       : {student.father_name}
Mother's Name       : {student.mother_name}

Father Profession   : {student.father_profession}
Mother Profession   : {student.mother_profession}

Blood Group         : {student.blood_group}

Favourite Hobby     : {student.hobby}

{student.nid_type}   : {student.nid_birth}

Date of Birth       : {student.date_of_birth}

Department          : {student.department}

Address             : {student.address}

Student Phone       : {student.student_phone}

Parent Phone        : {student.parent_phone}

Email               : {student.email}

CGPA                : {student.cgpa}

===================================
""")

                return

        print("❌ Student not found.")

    # ---------------- DELETE STUDENT ----------------
    def delete_student(self):

        delete_id = input("Enter Student ID to Delete: ")

        for student in self.students:

            if student.student_id == delete_id:

                self.students.remove(student)

                print("✅ Student Deleted Successfully!")

                return

        print("❌ Student not found.")

    # ---------------- SAVE DATA ----------------
    def save_data(self):

        try:
            data = []

            for student in self.students:

                data.append(student.to_dict())

            with open(self.filename, "w") as file:

                json.dump(data, file, indent=4)

            return True
        except Exception as e:
            print(f"❌ Error saving data: {e}")
            return False

    # ---------------- LOAD DATA ----------------
    def load_data(self):

        if not os.path.exists(self.filename):
            print("ℹ️ No existing data file found. Starting fresh.")
            return

        try:
            with open(self.filename, "r") as file:
                data = json.load(file)

                if not isinstance(data, list):
                    print("⚠️ Data file is corrupted. Starting fresh.")
                    return

                for item in data:
                    try:
                        # Handle old data that might not have nid_type
                        nid_type = item.get("nid_type", "NID")  # Default to NID for old data

                        student = Student(
                            student_id=item.get("student_id", ""),
                            name=item.get("name", ""),
                            father_name=item.get("father_name", item.get("fathername", "")),
                            mother_name=item.get("mother_name", item.get("mothername", "")),
                            father_profession=item.get("father_profession", ""),
                            mother_profession=item.get("mother_profession", ""),
                            blood_group=item.get("blood_group", ""),

                            hobby=item.get("hobby", ""),
                            nid_birth=item.get("nid_birth", ""),
                            nid_type=nid_type,
                            date_of_birth=item.get("date_of_birth", ""),
                            department=item.get("department", ""),
                            address=item.get("address", ""),
                            student_phone=item.get("student_phone", ""),
                            parent_phone=item.get("parent_phone", ""),
                            email=item.get("email", ""),
                            cgpa=float(item.get("cgpa", 0.0))
                        )
                        self.students.append(student)
                    except Exception as e:
                        print(f"⚠️ Skipping corrupted student record: {e}")
                        continue

            if len(self.students) > 0:
                print(f"✅ Successfully loaded {len(self.students)} student(s)")
            else:
                print("ℹ️ No valid student data found in file.")

        except json.JSONDecodeError:
            print("⚠️ JSON file is corrupted. Starting fresh.")
        except Exception as e:
            print(f"⚠️ Could not load previous data: {e}")
            print("ℹ️ Starting with empty database.")

    # ---------------- MAIN MENU ----------------
    def menu(self):

        while True:

            print("""
========== STUDENT MANAGEMENT SYSTEM ==========

1. Add Student
2. View Students
3. Search Student
4. Delete Student
5. Save & Exit

================================================
""")

            choice = input("Enter your choice (1-5): ")

            if choice == "1":

                self.add_student()

            elif choice == "2":

                self.view_students()

            elif choice == "3":

                self.search_student()

            elif choice == "4":

                self.delete_student()

            elif choice == "5":

                if self.save_data():
                    print("\n✅ Data Saved Successfully!")
                print("👋 Goodbye!")
                break

            else:

                print("❌ Invalid Choice! Please enter 1-5")


# ================= RUN SYSTEM =================

if __name__ == "__main__":
    system = StudentManagementSystem()
    system.menu()

✅ Successfully loaded 3 student(s)

========== STUDENT MANAGEMENT SYSTEM ==========

1. Add Student
2. View Students
3. Search Student
4. Delete Student
5. Save & Exit


Enter your choice (1-5): 1

========== ADD NEW STUDENT ==========

1. Enter Student ID: 1011
❌ Student ID already exists!
1. Enter Student ID: 10055
2. Enter Student Name: 4454
3. Enter Father's Name: 154
4. Enter Mother's Name: 
5. Enter Father's Profession: 
6. Enter Mother's Profession: 
7. Enter Blood Group (A+, B+, O+, AB+ etc.): 
8. Enter Favourite Hobby: 

9. Select Document Type:
   1. NID Card
   2. Birth Certificate
   Choose (1 or 2): 2
9. Enter Birth Certificate Number: 
❌ Birth Certificate number must contain only digits!
9. Enter Birth Certificate Number: 
❌ Birth Certificate number must contain only digits!
9. Enter Birth Certificate Number: 
❌ Birth Certificate number must contain only digits!
9. Enter Birth Certificate Number: 
❌ Birth Certificate number must contain only digits!
9. Enter Birth Certifi